In [ ]:
import sys
from pathlib import Path
import numpy as np
from transformers import WhisperProcessor

# Resolve project root directory from notebook location
notebook_path = Path.cwd()
project_root = notebook_path.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.evaluation.audio_eval import AudioDeafnessEvaluator
from src.models.loader import ModelEnvironmentLoader
from utils.memory import flush_memory, print_vram_usage

print("Audio environment setup completed.")

In [ ]:
# Configuration block targeting minimal VRAM footprint for Whisper
audio_config = {
    "model": {
        "model_name": "openai/whisper-base",
        "torch_dtype": "bfloat16",
        "device": "cuda",
    }
}

model_name = audio_config["model"]["model_name"]
loader = ModelEnvironmentLoader(audio_config)

# Load underlying model weights and audio feature extractor
whisper_model = loader.load_audio_model(model_name)
whisper_processor = WhisperProcessor.from_pretrained(model_name)

evaluator = AudioDeafnessEvaluator(whisper_model, whisper_processor)
print_vram_usage()

In [ ]:
# Create a dummy 1D audio waveform sequence simulating 3 seconds of sound at 16kHz
sampling_rate = 16000
duration = 3
dummy_audio = np.sin(
    2 * np.pi * 440 * np.arange(sampling_rate * duration) / sampling_rate
)

print("--- Running Baseline Audio Evaluation ---")
baseline_result = evaluator.evaluate_audio_array(
    dummy_audio, sampling_rate=sampling_rate
)

In [ ]:
print("--- Injecting Cortical Deafness Hook into Whisper Encoder ---")


def whisper_encoder_noise_hook(module, input_tensor, output_tensor):
    """Inject heavy noise variables into the audio hidden states during forward pass."""
    import torch

    # Whisper encoder returns a BaseModelOutput structure where index 0 holds hidden states
    if isinstance(output_tensor, torch.Tensor):
        noise = torch.randn_like(output_tensor) * 5.0
        return output_tensor + noise
    else:
        hidden_states = output_tensor[0]
        noise = torch.randn_like(hidden_states) * 5.0
        output_tensor[0] = hidden_states + noise
        return output_tensor


# Target the primary structural multi-modal audio encoder network
target_encoder = whisper_model.model.encoder
hook_handle = target_encoder.register_forward_hook(whisper_encoder_noise_hook)
print("Acoustic signal mutation hook is successfully mounted.")

In [ ]:
print("--- Running Post-Stroke Audio Evaluation ---")
try:
    post_stroke_result = evaluator.evaluate_audio_array(
        dummy_audio, sampling_rate=sampling_rate
    )
    print(
        f"\nToken Density Shift: {baseline_result['token_count']} -> {post_stroke_result['token_count']}"
    )
finally:
    # Safely detach hook constraints to prevent memory leakage
    hook_handle.remove()
    print("Acoustic mutation hook unmounted.")

In [ ]:
print("Terminating audio sequence session. Freeing GPU cells...")
del whisper_model
del whisper_processor
del evaluator

flush_memory()
print_vram_usage()